# Step 8c - Layer B (bias evaluation) and a test of the Layer A + B design

**What this tests.** There are no human labels on PESO, so the design cannot be scored against truth there. Three things can be checked:
1. **Internal consistency:** can a rule-based pre-annotator fill Layer A and Layer B for all 7,971 units so that every row passes the validator?
2. **Effect of the narrowed gender scope:** how the Step 9 silver labels (which mixed gender, age, marital, appearance) change under Layer B.
3. **External check:** on JABD (human labels), how well the pre-annotator finds gender references and gender bias, compared with the old Step 6/9 rules, using JABD labels restricted to gender.

**What this is not.** The pre-annotation is rule-made (silver). It is not a human label and does not validate the schema. Definitions of the rules below are my design choices, to be adjusted after your review.

## Layer B fields (as specified)

| Field | Values | Rule |
|---|---|---|
| `bias_type` | `explicit`, `implicit`, `none`, `unclear` | Final gender-bias classification. |
| `bias_presence` | `biased`, `not_biased`, `unclear` | Derived: explicit or implicit = biased; none = not_biased; unclear = unclear. |
| `bias_subtype` | `gender_exclusion`, `gender_preference`, `gender_stereotyping`, `gendered_role_language`, `none`, `unclear` | explicit uses exclusion or preference; implicit uses stereotyping or role language. |
| `rewrite_action` | `required`, `recommended`, `not_needed`, `manual_review` | Correction versus optional inclusion improvement. |
| `suggested_rewrite` | text | Replacement for the offending expression; required when `rewrite_action = required`. |

## Pre-annotation rules (Layer A type to Layer B), design choices

| Layer A type | bias_type | bias_subtype | rewrite_action | Reason |
|---|---|---|---|---|
| `restriction` | explicit | gender_exclusion | required | A stated single-gender requirement. |
| `preference` | explicit | gender_preference | required | One gender stated as preferred. |
| `binary_listing` ("Female/Male") | unclear | unclear | recommended | Not given the same class as "Female only" (your point 3); adjudication needed; inclusive rewrite is an improvement. |
| `gendered_title_pronoun` | implicit | gendered_role_language | recommended | Gendered job title or pronoun. |
| `gender_coded` | unclear | unclear | manual_review | Agentic or communal wording is not bias by default (your point 2); a human must judge the context. |
| `inclusive` | none | none | not_needed | Explicit inclusion statement. |
| `none` | none | none | not_needed | |
| gender mentioned but intent unclear | unclear | unclear | manual_review | |

A pre-annotator that never labels coded wording as bias will give **no** `implicit` stereotyping labels. This is intended, and it means implicit gender bias cannot be settled by rules.

In [1]:
import json, re, platform
from pathlib import Path
import numpy as np, pandas as pd, sklearn

SEED, N_BOOT = 42, 1000
OUT = Path("outputs/equal_step8c_layerB_test"); OUT.mkdir(parents=True, exist_ok=True)
SCHEMA_B = {
    "bias_type": ["explicit", "implicit", "none", "unclear"],
    "bias_presence": ["biased", "not_biased", "unclear"],
    "bias_subtype": ["gender_exclusion", "gender_preference", "gender_stereotyping", "gendered_role_language", "none", "unclear"],
    "rewrite_action": ["required", "recommended", "not_needed", "manual_review"],
}
(OUT / "step8c_layerB_schema.json").write_text(json.dumps(SCHEMA_B, indent=2), encoding="utf-8")
json.dump(dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, scikit_learn=sklearn.__version__, seed=SEED,
               bootstrap=f"{N_BOOT} resamples of job_id clusters"), open(OUT / "step8c_environment.json", "w"), indent=2)

## 1. Rule-based pre-annotator (Layer A then Layer B)

Gender words use the same lists as Step 6/9 (male, female, men, women and so on). Added here for gendered titles and single pronouns. The lemma option lets the same rules run on JABD, which is lemmatised ("woman", "man").

In [2]:
MASC = ("active adventurous aggress ambitio analy assert athlet autonom boast challeng compet confident "
        "courag decisive decision determin dominan driven fearless force headstrong hierarch impulsive "
        "independen individual intellect lead logic objective opinion outspoken persist self-relian "
        "self-sufficien stubborn superior unreasonab").split()
FEM = ("agree affectionate child cheer collab commit communal compassion connect considerate cooperat "
       "depend emotiona empath feel flatterable gentle honest interpersonal interdependen kind kinship "
       "loyal modest nurtur pleasant polite quiet respons sensitiv submissive support sympath tender "
       "together trust understand warm enthusias inclusive yield share").split()
FP = ("confidential", "committee", "shareholder", "kinds")
DUTY = ("analy", "lead", "respons", "support", "decision", "challeng", "collab", "cooperat")

def coded_hits(text, stems):
    out = []
    for m in re.finditer(r"[a-z][a-z-]*", text.lower()):
        w = m.group()
        if w.startswith(FP): continue
        ps = [w] + (w.split("-") if "-" in w else [])
        if any(p.startswith(s) for p in ps for s in stems): out.append((m.start(), m.end(), w))
    return out

MALE = {"std": r"male|males|men|gentlemen", "lem": r"male|males|men|man|gentlemen|gentleman"}
FEMALE = {"std": r"female|females|women|ladies", "lem": r"female|females|women|woman|ladies|lady"}
INCL_RE = re.compile(r"\b(?:either\s+gender|any\s+gender|regardless\s+of\s+(?:sex|gender)|men\s+and\s+women|"
                     r"gender\s*[:\-]?\s*(?:any|open|not\s+required))\b", re.I)
TITLE_RE = re.compile(r"\b(?:salesman|salesmen|saleslady|salesgirl|waiter|waitress|chairman|foreman|watchman|cameraman|busboy|houseboy|stewardess)\b", re.I)
PRON_BIN_RE = re.compile(r"\b(?:he\s*/\s*she|she\s*/\s*he|he\s+or\s+she|his\s*/\s*her|s/he)\b", re.I)
PRON_RE = re.compile(r"\b(?:he|she|his|her|him)\b", re.I)
ONLY_RE = re.compile(r"\b(?:only|exclusively|must\s+be|should\s+be|required)\b", re.I)
PREF_RE = re.compile(r"\b(?:prefer|preferably|preferred|preference)\b", re.I)

def layerA(text, lemma=False):
    k = "lem" if lemma else "std"
    m_re, f_re = re.compile(r"\b(?:%s)\b" % MALE[k], re.I), re.compile(r"\b(?:%s)\b" % FEMALE[k], re.I)
    m, f = m_re.search(text), f_re.search(text)
    n_words = len(re.findall(r"\w+", text))
    out = dict(gender_reference="no", gender_expression_type="none", gender_direction="na", gender_coding="na", span=None, context_dependent="no")
    inc, pb = INCL_RE.search(text), PRON_BIN_RE.search(text)
    if inc:
        out.update(gender_reference="yes", gender_expression_type="inclusive", gender_direction="multiple", span=inc.span())
    elif m and f:
        s = min(m.start(), f.start()); e = max(m.end(), f.end())
        out.update(gender_reference="yes", gender_expression_type="binary_listing", gender_direction="multiple", span=(s, e))
    elif m or f:
        h = m or f; d = "male" if m else "female"
        t = "restriction" if (n_words <= 4 or ONLY_RE.search(text)) and not PREF_RE.search(text) else ("preference" if PREF_RE.search(text) else "unclear")
        out.update(gender_reference="yes", gender_expression_type=t, gender_direction=d, span=h.span())
    elif pb:
        out.update(gender_reference="yes", gender_expression_type="binary_listing", gender_direction="multiple", span=pb.span())
    elif TITLE_RE.search(text) or PRON_RE.search(text):
        h = TITLE_RE.search(text) or PRON_RE.search(text)
        w = h.group().lower(); d = "male" if w in ("salesman", "salesmen", "waiter", "chairman", "foreman", "watchman", "cameraman", "busboy", "houseboy", "he", "his", "him") else "female"
        out.update(gender_reference="yes", gender_expression_type="gendered_title_pronoun", gender_direction=d, span=h.span())
    else:
        mh, fh = coded_hits(text, MASC), coded_hits(text, FEM)
        if mh or fh:
            allh = sorted(mh + fh)
            out.update(gender_expression_type="gender_coded", gender_coding="agentic" if mh and not fh else "communal" if fh and not mh else "mixed",
                       span=(allh[0][0], allh[0][1]), context_dependent="yes" if all(w.startswith(DUTY) for *_, w in allh) else "no")
    return out

BMAP = {  # Layer A type -> (bias_type, bias_subtype, rewrite_action)
    "restriction": ("explicit", "gender_exclusion", "required"), "preference": ("explicit", "gender_preference", "required"),
    "binary_listing": ("unclear", "unclear", "recommended"), "gendered_title_pronoun": ("implicit", "gendered_role_language", "recommended"),
    "gender_coded": ("unclear", "unclear", "manual_review"), "inclusive": ("none", "none", "not_needed"), "none": ("none", "none", "not_needed"),
    "unclear": ("unclear", "unclear", "manual_review")}
PRESENCE = {"explicit": "biased", "implicit": "biased", "none": "not_biased", "unclear": "unclear"}

def preannotate(text, lemma=False):
    a = layerA(text, lemma)
    bt, st, ra = BMAP[a["gender_expression_type"]]
    rewrite = "Open to all qualified candidates" if (bt == "explicit" or a["gender_expression_type"] == "binary_listing") and len(re.findall(r"\w+", text)) <= 4 else ""
    return dict(**a, bias_type=bt, bias_presence=PRESENCE[bt], bias_subtype=st, rewrite_action=ra, suggested_rewrite=rewrite)

for t in ["Male", "Female/Male", "Must be male, 25-30 years old", "Male preferably", "Salesman needed", "Strong leadership and competitive spirit",
          "Open to men and women", "Single, 18-25 years old", "With pleasing personality", "Graduate of BS Psychology"]:
    r = preannotate(t); print(f"{t!r:48} -> ref={r['gender_reference']:3} type={r['gender_expression_type']:22} bias={r['bias_type']:9} sub={r['bias_subtype']:22} rewrite={r['rewrite_action']}")

'Male'                                           -> ref=yes type=restriction            bias=explicit  sub=gender_exclusion       rewrite=required
'Female/Male'                                    -> ref=yes type=binary_listing         bias=unclear   sub=unclear                rewrite=recommended
'Must be male, 25-30 years old'                  -> ref=yes type=restriction            bias=explicit  sub=gender_exclusion       rewrite=required
'Male preferably'                                -> ref=yes type=preference             bias=explicit  sub=gender_preference      rewrite=required
'Salesman needed'                                -> ref=yes type=gendered_title_pronoun bias=implicit  sub=gendered_role_language rewrite=recommended
'Strong leadership and competitive spirit'       -> ref=no  type=gender_coded           bias=unclear   sub=unclear                rewrite=manual_review
'Open to men and women'                          -> ref=yes type=inclusive              bias=none      sub=

## 2. Layer A and B validator, applied to every pre-annotated PESO unit

In [3]:
A_REF_NO = {"gender_coded", "none", "unclear"}
def validate_AB(r):
    p = []
    t, ref, dr, gc = r["gender_expression_type"], r["gender_reference"], r["gender_direction"], r["gender_coding"]
    if ref == "no" and (t not in A_REF_NO or dr != "na"): p.append("A: reference=no inconsistent")
    if ref == "yes" and (t in A_REF_NO - {"unclear"} or dr == "na"): p.append("A: reference=yes inconsistent")
    if t == "binary_listing" and dr != "multiple": p.append("A: binary_listing direction")
    if (t == "gender_coded") != (gc != "na"): p.append("A: gender_coding")
    if t != "none":
        if r["span"] is None: p.append("A: span missing")
        else:
            s, e = r["span"]
            if not (0 <= s < e <= len(r["sentence"])): p.append("A: span offsets")
    for col in SCHEMA_B:
        if r[col] not in SCHEMA_B[col]: p.append(f"B: {col} value")
    if r["bias_presence"] != PRESENCE.get(r["bias_type"]): p.append("B: presence not derived from bias_type")
    if r["bias_type"] == "explicit" and r["bias_subtype"] not in ("gender_exclusion", "gender_preference"): p.append("B: explicit subtype")
    if r["bias_type"] == "implicit" and r["bias_subtype"] not in ("gender_stereotyping", "gendered_role_language"): p.append("B: implicit subtype")
    if r["bias_type"] == "none" and (r["bias_subtype"] != "none" or r["rewrite_action"] != "not_needed"): p.append("B: none inconsistent")
    if r["bias_type"] == "explicit" and r["rewrite_action"] not in ("required", "recommended"): p.append("B: explicit needs rewrite")
    if r["rewrite_action"] == "required" and not r["suggested_rewrite"]: p.append("B: required without suggested_rewrite")
    return p

units = pd.read_csv("outputs/equal_step4_sentence_embeddings/step4_sentence_units.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
silver = pd.read_csv("outputs/equal_step9_silver_baseline/step9_silver_labels.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
u = units.merge(silver[["sentence_id", "silver_bias_type"]], on="sentence_id", how="left", validate="one_to_one")
assert len(u) == 7971 and u.silver_bias_type.notna().all()
P = pd.DataFrame([preannotate(s) for s in u.sentence]); P["sentence"] = u.sentence.values
probs = P.apply(validate_AB, axis=1)
print("units pre-annotated:", len(P), "| units failing the validator:", int((probs.str.len() > 0).sum()))
print("\nproblems:", pd.Series([x for l in probs for x in l]).value_counts().to_dict() or "none")

units pre-annotated: 7971 | units failing the validator: 1

problems: {'B: required without suggested_rewrite': 1}


### 2.1 Required-rewrite units without a suggested rewrite

The validator above passes only when `required` units have a suggested rewrite. Rules fill the replacement only for very short gender-only units. Longer units need the Step 10 rules or a human; the pre-annotator marks them so the gap is visible.

In [4]:
print(P.rewrite_action.value_counts().to_string())
req = P[P.rewrite_action == "required"]
print("\nrequired:", len(req), "| with suggested_rewrite:", int((req.suggested_rewrite != "").sum()))

rewrite_action
not_needed       6929
manual_review     585
required          313
recommended       144

required: 313 | with suggested_rewrite: 312


## 3. PESO distributions under Layer A and B

In [5]:
dist = pd.concat({"Layer A type": P.gender_expression_type.value_counts(), "Layer B bias_type": P.bias_type.value_counts(),
                  "Layer B subtype": P.bias_subtype.value_counts(), "rewrite_action": P.rewrite_action.value_counts()}).rename("units")
display(dist.to_frame()); dist.to_csv(OUT / "step8c_peso_distributions.csv")
print("gender_reference = yes:", int((P.gender_reference == "yes").sum()), "of", len(P), f"({(P.gender_reference == 'yes').mean():.2%})")
P.assign(unit_id=u.sentence_id.values, ad_id=u.record_id.values, silver_step9=u.silver_bias_type.values).drop(columns=["span"]).to_csv(OUT / "step8c_peso_preannotation.csv", index=False)

units
Layer A type      none                     6929
                  gender_coded              577
                  restriction               299
                  binary_listing            106
                  gendered_title_pronoun     38
                  preference                 14
                  unclear                     8
Layer B bias_type none                     6929
                  unclear                   691
                  explicit                  313
                  implicit                   38
Layer B subtype   none                     6929
                  unclear                   691
                  gender_exclusion          299
                  gendered_role_language     38
                  gender_preference          14
rewrite_action    not_needed               6929
                  manual_review             585
                  required                  313
                  recommended               144

gender_reference = yes: 465 of 7971 (5.83%)


## 4. Effect of the narrowed scope: Step 9 silver label against Layer B

Rows are the Step 9 silver labels (which mixed gender, age, marital and appearance); columns are the Layer B bias type. A silver-explicit unit with Layer B `none` is mostly age or marital wording that is outside the gender scope.

In [6]:
ct = pd.crosstab(u.silver_bias_type.values, P.bias_type.values); display(ct); ct.to_csv(OUT / "step8c_silver_vs_layerB.csv")
se = u.silver_bias_type.values == "explicit"
print("Step 9 explicit units:", int(se.sum()), "| Layer B explicit:", int((P.bias_type[se] == "explicit").sum()), "| unclear:", int((P.bias_type[se] == "unclear").sum()), "| none:", int((P.bias_type[se] == "none").sum()))
print("\nStep 9 explicit units that Layer B calls none (sample of distinct texts):")
print(u.sentence[se & (P.bias_type.values == "none")].drop_duplicates().head(8).str[:100].to_string())
si = u.silver_bias_type.values == "implicit"
print("\nStep 9 implicit units:", int(si.sum()), "| Layer B explicit:", int((P.bias_type[si] == "explicit").sum()), "| unclear (coded wording):", int((P.bias_type[si] == "unclear").sum()), "| none:", int((P.bias_type[si] == "none").sum()))

col_0,explicit,implicit,none,unclear
row_0,,,,
explicit,313,0,61,110
implicit,0,1,88,579
none,0,37,6780,2


Step 9 explicit units: 484 | Layer B explicit: 313 | unclear: 110 | none: 61

Step 9 explicit units that Layer B calls none (sample of distinct texts):
147       18-30 years old
189       25-40 Years Old
375       25-30 Years old
389       18-25 years old
908    25 to 40 years old
912    21 to 27 years old
913                Single
921    21 to 30 years old

Step 9 implicit units: 668 | Layer B explicit: 0 | unclear (coded wording): 579 | none: 88


## 5. External check on JABD, restricted to gender

JABD gold is mapped to the narrowed scope: **gender biased** = a sentence carrying Explicit Gender Bias or Implicit Gender Bias; **gender reference** = a sentence carrying Explicit Marking of Sex, Generic He or Generic She. Age-only, ethnicity-only and the like count as negatives, because they are outside the gender model. The old Step 6/9 rules (gender, marital, age, appearance and coded wording all counting as bias) are the comparison.

Layer B can be scored two ways: **strict** counts only `explicit` and `implicit` as biased; **flagged** also counts `unclear`, which means "needs a human to decide".

Caveat: JABD's explicit gender class includes equal-opportunity statements and generic pronouns, which our schema treats as inclusive or unclear. Precision against JABD therefore understates how well the design does on our own definition.

In [7]:
JABD = Path("external/jabd/jabd_annotations_v2.csv.gz")
raw = pd.read_csv(JABD)
raw["text"] = raw.tokens.apply(lambda s: " ".join(json.loads(s)))
l2 = raw.text_labels2.apply(lambda s: set(json.loads(s))); l4 = raw.text_labels4.apply(lambda s: set(json.loads(s)))
raw["g_ref"] = l4.apply(lambda s: bool(s & {"Explicit Marking of Sex", "Generic He", "Generic She"}))
raw["g_bias"] = l2.apply(lambda s: bool(s & {"Explicit Gender Bias", "Implicit Gender Bias"}))
raw["g_expl"] = l2.apply(lambda s: "Explicit Gender Bias" in s)
print("JABD sentences:", len(raw), "| gender reference:", int(raw.g_ref.sum()), "| gender biased:", int(raw.g_bias.sum()), "| explicit gender:", int(raw.g_expl.sum()))

J = pd.DataFrame([preannotate(t, lemma=True) for t in raw.text])
raw["B_ref"] = (J.gender_reference == "yes").values
raw["B_strict"] = J.bias_type.isin(["explicit", "implicit"]).values
raw["B_flag"] = J.bias_type.isin(["explicit", "implicit", "unclear"]).values
raw["B_explicit"] = (J.bias_type == "explicit").values
# Old rules (Step 6/9 as in Step 12, lemma-adapted): any gender, marital, age, appearance or coded wording = biased
MARITAL_RE = re.compile(r"\b(?:single|married|unmarried|widow(?:ed|er)?|marital|pregnan\w*)\b", re.I)
AGE_RE = re.compile(r"\b(?:\d{2}\s*(?:-|to)\s*\d{2}\s*(?:years|yrs|yo|y/o|years\s+old)|(?:below|under|not\s+(?:more|older)\s+than|up\s+to|maximum\s+(?:age\s+)?of)\s*\d{2}|age[sd]?\s*(?:range|limit|requirement)?\s*[:\-]?\s*\d{2})\b", re.I)
APPEAR_RE = re.compile(r"\b(?:good\s+looking|pleasing\s+personality|presentable|attractive|height|weight|physically\s+fit|heavy\s+lifting|tall|slim)\b", re.I)
G_L = re.compile(r"\b(?:male|males|men|man|gentlemen|female|females|women|woman|ladies|lady)\b", re.I)
def old_biased(t):
    return bool((G_L.search(t) and not INCL_RE.search(t)) or MARITAL_RE.search(t) or AGE_RE.search(t) or APPEAR_RE.search(t) or coded_hits(t, MASC) or coded_hits(t, FEM))
raw["old_biased"] = raw.text.apply(old_biased)

JABD sentences: 14960 | gender reference: 1685 | gender biased: 8802 | explicit gender: 1685


In [8]:
jobs = raw.job_id.values; uj, inv = np.unique(jobs, return_inverse=True)
rng = np.random.default_rng(SEED); W = rng.multinomial(len(uj), np.ones(len(uj)) / len(uj), size=N_BOOT)
def cnt(gold, pred):
    c = np.zeros((len(uj), 3)); np.add.at(c[:, 0], inv, gold & pred); np.add.at(c[:, 1], inv, ~gold & pred); np.add.at(c[:, 2], inv, gold & ~pred); return c
def prf(c):
    tp, fp, fn = c; p = tp / (tp + fp) if tp + fp else np.nan; r = tp / (tp + fn) if tp + fn else np.nan
    return p, r, (2 * p * r / (p + r) if p == p and r == r and p + r else 0.0)
def row(task, system, gold, pred):
    c = cnt(gold.values, pred.values); p, r, f = prf(c.sum(0)); bs = np.array([prf(w @ c) for w in W])
    ci = lambda k: f"{np.nanpercentile(bs[:, k], 2.5):.3f}-{np.nanpercentile(bs[:, k], 97.5):.3f}"
    return dict(task=task, system=system, flagged=int(pred.sum()), precision=round(p, 3), precision_CI=ci(0), recall=round(r, 3), recall_CI=ci(1), F1=round(f, 3), F1_CI=ci(2))
R = pd.DataFrame([
    row("gender reference", "Layer A pre-annotator", raw.g_ref, raw.B_ref),
    row("gender biased", "old Step 6/9 rules", raw.g_bias, raw.old_biased),
    row("gender biased", "Layer B strict (explicit+implicit)", raw.g_bias, raw.B_strict),
    row("gender biased", "Layer B flagged (+unclear)", raw.g_bias, raw.B_flag),
    row("explicit gender bias", "Layer B explicit", raw.g_expl, raw.B_explicit)])
R.to_csv(OUT / "step8c_jabd_layerB_results.csv", index=False); display(R)
print("prevalence of gender-biased in JABD:", round(raw.g_bias.mean(), 3), "(precision of a random flag equals this)")

,task,system,flagged,precision,precision_CI,recall,recall_CI,F1,F1_CI
0,gender reference,Layer A pre-annotator,1004,0.880,0.860-0.900,0.525,0.492-0.556,0.657,0.630-0.684
1,gender biased,old Step 6/9 rules,5037,0.814,0.804-0.825,0.466,0.455-0.477,0.593,0.583-0.603
2,gender biased,Layer B strict (explicit+implicit),608,0.931,0.909-0.951,0.064,0.057-0.072,0.120,0.107-0.134
3,gender biased,Layer B flagged (+unclear),5398,0.825,0.815-0.835,0.506,0.494-0.517,0.627,0.617-0.637
4,explicit gender bias,Layer B explicit,14,0.857,0.667-1.000,0.007,0.004-0.011,0.014,0.007-0.022


prevalence of gender-biased in JABD: 0.588 (precision of a random flag equals this)


### 5.1 How JABD's explicit-gender sentences look under Layer A

This shows where the definitions differ: how many JABD explicit-gender sentences fall into each Layer A type, and what the pre-annotator misses.

In [9]:
J["jabd_expl_gender"] = raw.g_expl.values
t = J[J.jabd_expl_gender].gender_expression_type.value_counts().rename("JABD explicit-gender sentences").to_frame(); display(t); t.to_csv(OUT / "step8c_jabd_explicit_by_layerA.csv")
miss = raw[raw.g_expl & ~raw.B_flag]
print("JABD explicit-gender sentences with no Layer B flag:", len(miss), "of", int(raw.g_expl.sum()))
print(miss.text.str[:120].sample(8, random_state=SEED).to_string())
print("\nLayer B explicit flags that are NOT gender-biased in JABD (false positives):")
fp = raw[raw.B_explicit & ~raw.g_bias]; print(len(fp)); print(fp.text.str[:120].head(8).to_string())

,JABD explicit-gender sentences
gender_expression_type,
gendered_title_pronoun,539
none,508
gender_coded,293
binary_listing,203
unclear,130
restriction,7
preference,5


JABD explicit-gender sentences with no Layer B flag: 508 of 1685
955      fastsigns® be an equal opportunity employer th...
5849     we take every measure to carefully ensure our ...
13474    pierre heafey have surround himself with a tea...
8440     we take every measure to carefully ensure our ...
3700     please submit your resume and salary requireme...
14467    this job may be for you if you want to be on a...
805      ) we be an equal opportunity employer and all ...
3890     tidewater finance company be an equal opportun...

Layer B explicit flags that are NOT gender-biased in JABD (false positives):
2
733                     she will prefer to hire a female .
10268    please note that female applicant be strongly ...


## 6. Reading guide and limits
- Everything on PESO is rule-made. Consistency (section 2) shows the schema can be applied without contradictions; it does not show that annotators will agree or that the labels are right.
- The pre-annotator never labels coded wording as bias, so `implicit` stereotyping comes out empty by design. That is a statement about rules, not about the corpus.
- The JABD check is a transfer test on a different population and a different definition (see section 5). Precision there depends on JABD's enriched prevalence.
- `binary_listing` is `unclear` with a recommended rewrite. This follows your note that "Female/Male" should not receive the same class as "Female only", but it leaves the bias verdict to an adjudicator.
- Step 9, 10 and 11 are not changed by this notebook.